<a href="https://colab.research.google.com/github/sophiashiuu/DS340-Final-Project/blob/main/ensemble_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
from google.colab import drive
import numpy as np
import pandas as pd

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


# Load data

In [3]:
folder_path = "/content/drive/MyDrive/DS 340/Final Project/"

X = np.load(folder_path + "X_lyrics_filtered.npy")
y = np.load(folder_path + "y_lyrics_filtered.npy")
music_ids = np.load(folder_path + "music_ids_filtered.npy")

In [5]:
print(y)

[1 0 2 0 0 3 3 3 3 3 3 3 3 1 2 3 3 1 3 3 3 1 3 3 3 3 3 3 3 3 2 3 2 0 2 0 3
 0 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 2 3 3 3 3 0 3 0 3 3 3 3 3 3 3 3 3 1
 3 0 3 3 3 3 3 2 3 3 3 3 3 3 3 3 3 1 0 3 1 3 3 3 3 3 3 3 3 3 3 3 3 3 1 3 3
 1 0 3 3 2 3 1 3 2 3 3 3 0 0 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 3 2 3 1 0 2 0
 1 0 0 0 1 0 2 3 3 3 0 2 0 0 3 0 0 0 3 0 3 0 0 0 0 1 3 0 3 3 3 3 3 3 3 3 1
 3 3 3 3 3 3 3 0 1 3 3 3 3 1 3 3 3 3 3 3 3 3 3 3 3 3 0 0 0 0 1 3 1 0 1 3 3
 0 0 0 2 3 0 2 3 3 3 3 2 3 3 0 3 2 3 3 0 0 2 0 3 3 3 3 2 3 3 1 0 3 3 3 3 3
 3 1 0 3 0 1 2 0 0 0 0 3 3 3 0 3 3 3 3 3 3 3 3 3 3 1 1 1 0 3 3 3 3 3 3 3 3
 3 3 3 3 3 2 3 3 2 3 2 3 2 3 3 0 3 1 3 3 0 3 3 0 0 1 2 1 0 2 3 3 3 3 2 3 1
 0 0 0 3 3 3 3 3 3 3 2 3 3 3 3 3 3 3 3 3 3 3 0 3 2 1 3 2 0 0 0 3 3 3 1 3 3
 3 3 3 3 3 1 2 0 3 0 3 1 3 3 3 3 3 3 3 3 3 0 3 3 3 3 3 3 3 3 3 0 3 3 3 3 0
 3 0 3 3 3 3 3 0 2 2 2 2 2 3 3 3 3 1 3 3 0 3 2 0 2 0 3 0 1 0 3 3 3 1 3 3 3
 3 3 3 2 3 3 3 3 3 0 3 3 3 3 3 3 3 0 2 0 3 3 3 3 3 2 3 3 3 1 3 1 3 1 3 3 3
 0 3 2 3 2 2 0 0 2 3 3 3 

# Create valence, arousal labels

In [27]:
y_valence = []
y_arousal = []

for label in y:
    # Valence
    if label in [1, 3]:
        y_valence.append(1)
    else:
        y_valence.append(0)
    # Arousal
    if label in [2, 3]:
        y_arousal.append(1)
    else:
        y_arousal.append(0)

y_valence = np.array(y_valence)
y_arousal = np.array(y_arousal)

In [28]:
print("Valence:", np.unique(y_valence, return_counts=True))
print("Arousal:", np.unique(y_arousal, return_counts=True))

Valence: (array([0, 1]), array([160, 445]))
Arousal: (array([0, 1]), array([151, 454]))


# Split data: specific indices to avoid data leakage

In [31]:
from sklearn.model_selection import train_test_split

In [8]:
indices = np.arange(len(X))

train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

In [9]:
X_train = X[train_idx]
X_test = X[test_idx]

y_valence_train = y_valence[train_idx]
y_valence_test = y_valence[test_idx]

y_arousal_train = y_arousal[train_idx]
y_arousal_test = y_arousal[test_idx]

In [10]:
np.save(folder_path + "train_music_ids.npy", music_ids[train_idx])
np.save(folder_path + "test_music_ids.npy", music_ids[test_idx])

# Create logistic regression models

In [38]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import classification_report

In [33]:
valence_model = LogisticRegression(class_weight="balanced")
arousal_model = LogisticRegression(class_weight="balanced")

In [34]:
valence_model.fit(X_train, y_valence_train)
arousal_model.fit(X_train, y_arousal_train)

LogisticRegression(class_weight='balanced')

In [35]:
valence_pred = valence_model.predict(X_test)
arousal_pred = arousal_model.predict(X_test)

In [37]:
print("Valence balanced accuracy:", balanced_accuracy_score(y_valence_test, valence_pred))
print("Arousal balanced accuracy:", balanced_accuracy_score(y_arousal_test, arousal_pred))

Valence balanced accuracy: 0.7357794943820224
Arousal balanced accuracy: 0.5465201465201466


In [39]:
print("Valence classification report:")
print(classification_report(y_valence_test, valence_pred))

print("Arousal classification report:")
print(classification_report(y_arousal_test, arousal_pred))

Valence classification report:
              precision    recall  f1-score   support

           0       0.51      0.72      0.60        32
           1       0.88      0.75      0.81        89

    accuracy                           0.74       121
   macro avg       0.70      0.74      0.70       121
weighted avg       0.78      0.74      0.76       121

Arousal classification report:
              precision    recall  f1-score   support

           0       0.29      0.47      0.36        30
           1       0.78      0.63      0.70        91

    accuracy                           0.59       121
   macro avg       0.54      0.55      0.53       121
weighted avg       0.66      0.59      0.61       121



# Use GridSearch to find optimal hyperparameters

In [40]:
from sklearn.model_selection import GridSearchCV

In [43]:
param_grid = {"C": [0.01, 0.1, 1, 10, 100]}

valence_search = GridSearchCV(LogisticRegression(class_weight="balanced"), param_grid, cv=5, scoring="balanced_accuracy")
valence_search.fit(X_train, y_valence_train)

arousal_search = GridSearchCV(LogisticRegression(class_weight="balanced"), param_grid, cv=5, scoring="balanced_accuracy")
arousal_search.fit(X_train, y_arousal_train)

print("Valence:")
print("Best C:", valence_search.best_params_)
print("Best CV accuracy:", valence_search.best_score_)

print("Arousal:")
print("Best C:", arousal_search.best_params_)
print("Best CV accuracy:", arousal_search.best_score_)

Valence:
Best C: {'C': 0.01}
Best CV accuracy: 0.7082478632478632
Arousal:
Best C: {'C': 0.01}
Best CV accuracy: 0.6042922374429224


In [44]:
valence_model = valence_search.best_estimator_
arousal_model = arousal_search.best_estimator_